<div align="center">
    <img src="assets/neovim-logo.svg" width="260" alt="Neovim"/>
</div>

# Neovim Essentials: Day 2 - Lazyvim, and hands-on

On Day 02, we will install Lazyvim and get hands on in a project.

## 💤 LazyVim

### What is it?

[LazyVim](https://www.lazyvim.org/) is a **Neovim distribution**: a ready-made configuration that turns plain nvim into a full IDE-like editor out of the box. It was created by [folke](https://github.com/folke), who also wrote many popular Neovim plugins.

It gives you, preconfigured:

- A file explorer, fuzzy finder (files, text, buffers), and a start dashboard.
- Syntax highlighting via Treesitter, and LSP support (autocompletion, go-to-definition, diagnostics) with language servers installed through [Mason](https://github.com/mason-org/mason.nvim).
- Git integration, a status line, keymap hints (press `<Space>` and wait), and a polished theme.
- **Language extras**: one-line opt-ins for Python, Rust, Go, etc., via the `:LazyExtras` menu.

A few things that often cause confusion:

- **LazyVim ≠ lazy.nvim.** [lazy.nvim](https://github.com/folke/lazy.nvim) is the *plugin manager* that installs and loads plugins. LazyVim is a *configuration* built on top of it.
- **It's still your config.** LazyVim lives in `~/.config/nvim` like any other config. You can override any default and add your own plugins in `lua/plugins/`.
- **It's a good starting point, not the finish line.** As mentioned on Day 1, nvim is something you make your own. LazyVim gets you a usable editor on day one while you learn what you actually want.

### Requirements

- A recent Neovim. Check the exact minimum version in the [LazyVim installation docs](https://www.lazyvim.org/installation), and compare it with `nvim --version`.
- `git`
- A [Nerd Font](https://www.nerdfonts.com/) set as your terminal font (optional, but needed for icons).
- A C compiler (for Treesitter). On macOS, `xcode-select --install` provides one.
- Recommended: `ripgrep`, `fd`, `fzf` (for searching) and `lazygit` (git UI). On macOS:

```bash
brew install git ripgrep fd fzf lazygit
```

### Step 1: Check for an existing config

> ⚠️ LazyVim installs into `~/.config/nvim`. If you already have something there, back it up first rather than overwriting it:

```bash
ls ~/.config/nvim
```

- `No such file or directory`: you have no config yet. Skip to Step 2.
- Files are listed: back them up, along with nvim's data, state and cache folders (these hold plugins and history tied to the old config):

```bash
mv ~/.config/nvim{,.bak}

# optional but recommended
mv ~/.local/share/nvim{,.bak}
mv ~/.local/state/nvim{,.bak}
mv ~/.cache/nvim{,.bak}
```

### Step 2: Install the LazyVim starter

```bash
git clone https://github.com/LazyVim/starter ~/.config/nvim
rm -rf ~/.config/nvim/.git
```

Removing `.git` disconnects the folder from the starter repository, so you can put it under your own version control later.

### Step 3: Launch and check health

```bash
nvim
```

On first launch, lazy.nvim downloads and installs all the plugins. Wait for it to finish, then restart nvim. Afterwards, run:

```vim
:LazyHealth
```

This checks that everything (Neovim version, git, compiler, ripgrep, etc.) is in place, and tells you what's missing.

### Useful commands to start with

| Command / key | What it does |
|---|---|
| `<Space>` (wait) | Shows available keymaps |
| `:Lazy` | Plugin manager UI (update, inspect, clean plugins) |
| `:LazyExtras` | Enable language support and optional features |
| `:Mason` | Install language servers, formatters and linters |
| `<Space><Space>` | Find files |
| `<Space>/` | Search text in the project |
| `<Space>e` | Toggle file explorer |

## Setting up the Weather-LLM project

On Day 02, we will be working on an application that demonstrates the ["Tool Use" pattern](https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview) of Agentic AI. It is a well-known pattern for developing agentic AI applications where the agent - through a programming interface - asks the program itself to run certain tools and send the responses back.

### Why do we need tools?

An LLM is trained on a fixed snapshot of data. It knows a lot *about* the weather in London, but it has no idea what the temperature is *right now*: that information did not exist when the model was trained, and the model has no way of looking it up by itself.

Tools close that gap. A **tool** is just a function in your program (here, `get_temperature`) that you describe to the model. When the model decides it needs live data, it asks your program to run that function, and you hand the result back so that the model can use it in its answer.

### How the model learns about a tool

Along with the user's message, your code sends the model a **tool schema**: the tool's name, a description of what it does, and the arguments it takes along with their types. For `get_temperature`, that is roughly:

```json
{
  "name": "get_temperature",
  "description": "Get the current temperature for a city",
  "parameters": {
    "type": "object",
    "properties": {
      "city": { "type": "string", "description": "The name of the city" }
    },
    "required": ["city"]
  }
}
```

In the weather-llm project you don't write this JSON by hand: the Python library builds it from the function's **name, type hints and docstring**. This means the docstring is not just a comment for humans, it is what the model reads to decide when (and how) to call the tool. A vague docstring leads to a model that calls the tool at the wrong time, or not at all.

### The request/response loop

<div align="center">
    <img src="assets/day-2-app.png" width="950" class="diagram" alt="Sequence diagram of the tool use loop between the user, weather-llm, the LLM and Open-Meteo"/>
</div>

Read the diagram as three distinct responsibilities:

- **Steps 2-3: the model decides.** It sees the tool schema and replies with a *request* to call `get_temperature` with `city="London"`. This is not a result: the model only names the function and its arguments. It cannot execute code, and it never touches the network.
- **Steps 4-7: your code executes.** Nothing happens unless your program reads that request, calls the real Python function with the model's arguments, calls the [Open-Meteo](https://open-meteo.com/) API, and appends the result to the conversation as a *tool result* message.
- **Steps 8-9: the model explains.** With the tool result now in the conversation, the model is asked again, and only this time can it phrase an answer about today's weather.

A few things worth keeping in mind:

- **The conversation is stateless.** Every request sends the *whole* message history again: the user's question, the model's tool request, and the tool result. The model does not remember the previous call.
- **The loop can repeat.** A model may ask for several tool calls, or the same tool with different arguments (e.g. "compare London and Paris"), before it has enough to answer. A robust agent keeps looping until the model replies with plain text instead of a tool request.
- **Tools can fail.** If the city does not exist or the API is down, your code decides what to send back. Returning a clear error message as the tool result lets the model explain the problem to the user instead of making up a temperature.
- **You stay in control.** Because your code runs the tool, you can validate the arguments, limit what the tool is allowed to do, or ask the user for confirmation before doing something risky.

In the weather-llm project, all of this lives in `src/weather_llm/__init__.py`: `get_temperature` is the tool, and `ask` runs the loop.

### Cloning the repository and creating a virtual environment

The project lives in its own repository: [dasunpubudumal/weather-llm](https://github.com/dasunpubudumal/weather-llm). Clone it somewhere outside this `nvim-essentials` folder, and move into it:

```bash
git clone https://github.com/dasunpubudumal/weather-llm.git
cd weather-llm
```

The project requires **Python 3.14 or newer** (see `requires-python` in `pyproject.toml`). Check your version with `python3 --version`.

A **virtual environment** (venv) is a folder (`.venv`) holding a private Python interpreter and the project's packages. It keeps this project's dependencies separate from your system Python and from your other projects. There are two ways of creating one.

#### Option 1: `uv` (recommended)

The project is managed with [`uv`](https://docs.astral.sh/uv/), and comes with a `uv.lock` file that pins the exact version of every dependency. Install `uv` if you don't have it:

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

Then, from inside the `weather-llm` folder:

```bash
uv sync
```

`uv sync` creates `.venv`, downloads Python 3.14 if you don't have it, and installs the exact dependency versions from `uv.lock`. You don't need to activate the venv to run the project: `uv run` uses it automatically.

```bash
uv run weather-llm
```

#### Option 2: `venv` and `pip`

If you'd rather use only the standard Python tools, and you have Python 3.14 installed:

```bash
python3.14 -m venv .venv
source .venv/bin/activate
pip install -e .
```

`pip install -e .` installs the project in *editable* mode, along with the dependencies listed in `pyproject.toml`, so the changes you make in Neovim are picked up without reinstalling. Once installed, run it with:

```bash
weather-llm
```

On Windows (PowerShell), use `py -3.14 -m venv .venv` and then `.venv\Scripts\Activate.ps1`. Run `deactivate` to leave the venv.

> 💡 Open the project with `nvim .` from inside the `weather-llm` folder. LazyVim's Python language server looks for a `.venv` in the project root, so creating the venv *before* opening Neovim means imports such as `requests` resolve and you get completions and go-to-definition for them.

The application also needs a model to talk to. Follow the [weather-llm README](https://github.com/dasunpubudumal/weather-llm#readme) to set up the model (or to run everything in containers with Podman instead).